# Panel de préstamos y depósitos por provincia (distribgeo)

Apila los 133 archivos `Préstamos y depósitos MMaaaa.xlsx` ubicados en `Tec_Cont/distribgeo/` de cada dump IEF. Cada archivo es un snapshot trimestral con préstamos y depósitos por entidad × provincia.

Output: `data/interim/paneles_largos/panel_distribgeo.parquet` con columnas:
- `codigo_entidad`, `nombre_entidad`
- `fecha_corte` (el trimestre al que se refiere el dato)
- `provincia` (literal del BCRA)
- `prestamos`, `depositos` (en miles de pesos)
- `dump_yyyymm` (mes del dump del que vino la observación)

Decisión: cuando una observación `(entidad, fecha_corte, provincia)` aparece en múltiples dumps (es lo normal: el corte trimestral aparece en 3 dumps consecutivos), me quedo con la versión del dump más reciente (que puede tener correcciones).

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "utils"))
from paths import RAW, INTERIM, DIMENSIONES, PANELES, REPO

import pandas as pd
import duckdb
import re

IEF_ROOT = RAW / "bcra_ief"
OUT = PANELES / "panel_distribgeo.parquet"
OUT.parent.mkdir(parents=True, exist_ok=True)

## Lectura y apilado

Itero por cada dump (carpeta yyyymm), busco el xlsx en `Tec_Cont/distribgeo/` y lo apilo. El nombre del archivo varía a veces (capitalización, acentos), por eso uso glob.

In [ ]:
dumps = sorted([d for d in IEF_ROOT.iterdir() if d.is_dir() and d.name.isdigit()])
print(f"Dumps encontrados: {len(dumps)}")

bloques = []
for d in dumps:
    yyyymm_dump = d.name
    xlsx_files = list((d / "Entfin/Tec_Cont/distribgeo").glob("*.xlsx"))
    if not xlsx_files:
        continue
    # Las primeras 3 filas del xlsx son títulos/notas; los headers reales están en la fila 4 (header=3)
    df = pd.read_excel(xlsx_files[0], header=3)
    df["dump_yyyymm"] = int(yyyymm_dump)
    bloques.append(df)

raw = pd.concat(bloques, ignore_index=True)
print(f"Filas apiladas: {len(raw):,}")
print(f"Columnas: {raw.columns.tolist()}")
raw.head()

## Normalización de columnas

Los nombres de columnas vienen del BCRA con acentos, espacios y notas al pie. Los uniformo a snake_case.

In [ ]:
rename = {
    "Fecha": "fecha_corte_int",
    "Código": "codigo_entidad",
    "Entidad": "nombre_entidad",
    "Provincia": "provincia",
    "Préstamos (1)": "prestamos_miles_pesos",
    "Depósitos (2)": "depositos_miles_pesos",
}
raw = raw.rename(columns=rename)
# Filtro: la fecha tiene que ser parseable como int de 8 dígitos. Quedan afuera las filas
# de notas al pie y subtotales que algunos xlsx incluyen.
fecha_num = pd.to_numeric(raw["fecha_corte_int"], errors="coerce")
raw = raw[fecha_num.notna() & raw["codigo_entidad"].notna()].copy()
raw["fecha_corte_int"] = fecha_num.dropna().astype(int)

# El código viene como float/int (ej. 7), lo formateo al string de 5 dígitos del BCRA
raw["codigo_entidad"] = pd.to_numeric(raw["codigo_entidad"], errors="coerce").astype("Int64").astype(str).str.zfill(5)
raw["fecha_corte"] = pd.to_datetime(raw["fecha_corte_int"].astype(str), format="%Y%m%d", errors="coerce")
raw["yyyymm_corte"] = raw["fecha_corte"].dt.strftime("%Y%m").astype(int)
raw["prestamos"] = raw["prestamos_miles_pesos"] * 1000
raw["depositos"] = raw["depositos_miles_pesos"] * 1000

raw = raw[["codigo_entidad", "nombre_entidad", "yyyymm_corte", "fecha_corte",
           "provincia", "prestamos", "depositos", "dump_yyyymm"]]
print(f"Cortes distintos: {raw['yyyymm_corte'].nunique()}")
print(f"Rango: {raw['yyyymm_corte'].min()} - {raw['yyyymm_corte'].max()}")

## Deduplicación

Para cada `(codigo_entidad, yyyymm_corte, provincia)` me quedo con la observación del `dump_yyyymm` más alto.

In [ ]:
raw = raw.sort_values(["codigo_entidad", "yyyymm_corte", "provincia", "dump_yyyymm"])
panel = raw.drop_duplicates(subset=["codigo_entidad", "yyyymm_corte", "provincia"], keep="last")
print(f"Filas tras deduplicar: {len(panel):,}")

## Persistencia

In [ ]:
panel.to_parquet(OUT, index=False)
print(f"Escrito: {OUT.relative_to(REPO)}")
print(f"Filas: {len(panel):,}")

## Validaciones

In [ ]:
assert panel[["codigo_entidad", "yyyymm_corte", "provincia"]].duplicated().sum() == 0
assert panel["prestamos"].notna().any()
assert panel["depositos"].notna().any()

resumen = {
    "filas": len(panel),
    "entidades": panel["codigo_entidad"].nunique(),
    "provincias": panel["provincia"].nunique(),
    "cortes_trimestrales": panel["yyyymm_corte"].nunique(),
    "fecha_min": panel["yyyymm_corte"].min(),
    "fecha_max": panel["yyyymm_corte"].max(),
}
print("Validaciones OK")
for k, v in resumen.items():
    print(f"  {k}: {v:,}")

## Muestra: provincias por banco - último trimestre

In [ ]:
duckdb.sql(f"""
    select codigo_entidad, nombre_entidad, provincia,
           prestamos / 1e9 as prestamos_miles_millones,
           depositos / 1e9 as depositos_miles_millones
    from '{OUT}'
    where codigo_entidad = '00007' and yyyymm_corte = (select max(yyyymm_corte) from '{OUT}')
    order by depositos desc
    limit 10
""").df()